In [3]:
# filter dataset
!python MT-Preparation/filtering/filter.py ./en-zh.en ./en-zh.zh en zh

Dataframe shape (rows, columns): (231267, 2)
--- Rows with Empty Cells Deleted	--> Rows: 231267
--- Duplicates Deleted			--> Rows: 229646
--- Source-Copied Rows Deleted		--> Rows: 229640
--- Too Long Source/Target Deleted	--> Rows: 224743
--- HTML Removed			--> Rows: 224743
--- Rows will remain true-cased		--> Rows: 224743
--- Rows with Empty Cells Deleted	--> Rows: 224743
--- Source Saved: ./en-zh.en-filtered-wsd.en
--- Target Saved: ./en-zh.zh-filtered-wsd.zh


## Perform BERT-WSD on SoC Computer Cluster

1. **SSH to your SoC Computer Cluster**  

2. **Run `salloc` to acquire a GPU host:**  
   ```bash
   salloc -G nv

3. **Enter the host `slurm`:** 
    ```bash
    srun --pty bash

4. **Find number of CPUs/cores on the machine:**
    ```bash
    nproc --all
(Adjust batch size in `preprocess_file` function to match the number of CPUs)

5. **Run the file to perform WSD using BERT:**
    ```bash
    nice -n 400 python wsd.py

In [10]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
import nltk
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize, TreebankWordDetokenizer
import os

# Download necessary NLTK resources
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('wordnet')

class BertWSDProcessor:
    def __init__(self, model_path='bert-base-uncased', device='cuda' if torch.cuda.is_available() else 'cpu'):
        self.device = device
        self.tokenizer = BertTokenizer.from_pretrained(model_path)
        self.model = BertForSequenceClassification.from_pretrained(model_path)
        self.model.to(self.device)
        self.model.eval()

    def identify_ambiguous_words(self, sentence):
        tokens = word_tokenize(sentence)
        return [token for token in tokens if len(wordnet.synsets(token)) > 1]

    def disambiguate_batch(self, batch_sentences, ambiguous_per_sentence):
        inputs = self.tokenizer(batch_sentences, padding=True, truncation=True, return_tensors="pt").to(self.device)
        with torch.no_grad():
            logits = self.model(**inputs).logits
        preds = torch.argmax(logits, dim=1).tolist()

        results = []
        for sent, ambig_words, sense_id in zip(batch_sentences, ambiguous_per_sentence, preds):
            tokens = word_tokenize(sent)
            new_tokens = []
            used = set()
            for tok in tokens:
                if tok in ambig_words and tok not in used:
                    new_tokens.append(f"{tok}#{sense_id}")
                    used.add(tok)
                else:
                    new_tokens.append(tok)
            results.append(TreebankWordDetokenizer().detokenize(new_tokens))
        return results

def preprocess_file(input_file, output_file, batch_size=32, save_interval=1000):
    processor = BertWSDProcessor()
    processed_lines_count = 0

    if os.path.exists(output_file):
        with open(output_file, 'r', encoding='utf-8') as f:
            processed_lines_count = 0
    
    print(f"Resuming from line {processed_lines_count}...")

    with open(input_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()[processed_lines_count:]

    buffer = []

    for i in range(0, len(lines), batch_size):
        batch = [line.strip() for line in lines[i:i+batch_size]]
        ambig_map = [processor.identify_ambiguous_words(s) for s in batch]

        if any(ambig_map):  # At least one ambiguous word exists
            disambiguated = processor.disambiguate_batch(batch, ambig_map)
        else:
            disambiguated = batch

        buffer.extend(disambiguated)
        print(f"Processed {processed_lines_count + min(i+batch_size, len(lines))}/{processed_lines_count + len(lines)} lines")

        if len(buffer) >= save_interval:
            with open(output_file, 'a', encoding='utf-8') as f:
                f.write('\n'.join(buffer) + '\n')
            print(f"Saved {len(buffer)} lines to {output_file}")
            buffer = []

    if buffer:
        with open(output_file, 'a', encoding='utf-8') as f:
            f.write('\n'.join(buffer) + '\n')
        print(f"Final save: {len(buffer)} lines to {output_file}")

    print(f"Preprocessing complete. Output saved to {output_file}")

if __name__ == "__main__":
    input_file = "./en-zh.en-filtered-wsd.en"
    output_file = "./en-zh.en-filtered-wsd-processed.en"
    
    preprocess_file(input_file, output_file)


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Resuming from line 0...
Processed 32/224743 lines
Processed 64/224743 lines
Processed 96/224743 lines
Processed 128/224743 lines
Processed 160/224743 lines
Processed 192/224743 lines
Processed 224/224743 lines
Processed 256/224743 lines
Processed 288/224743 lines
Processed 320/224743 lines
Processed 352/224743 lines
Processed 384/224743 lines
Processed 416/224743 lines
Processed 448/224743 lines
Processed 480/224743 lines
Processed 512/224743 lines
Processed 544/224743 lines
Processed 576/224743 lines
Processed 608/224743 lines
Processed 640/224743 lines
Processed 672/224743 lines
Processed 704/224743 lines
Processed 736/224743 lines
Processed 768/224743 lines
Processed 800/224743 lines
Processed 832/224743 lines
Processed 864/224743 lines
Processed 896/224743 lines
Processed 928/224743 lines
Processed 960/224743 lines
Processed 992/224743 lines
Processed 1024/224743 lines
Saved 1024 lines to ./en-zh.en-filtered-wsd-processed.en
Processed 1056/224743 lines
Processed 1088/224743 lines
P

In [28]:
import numpy as np
import yake
import multiprocessing as mp
from tqdm import tqdm


def extract_yake_chunk(chunk_indices, corpus, window_size, top_k, worker_id=None):
    kw_extractor = yake.KeywordExtractor(lan="en", n=1, top=top_k)
    chunk_result = []

    for i in chunk_indices:
        if i < window_size // 2:
            num_before = i
            num_after = window_size - num_before
        elif i > len(corpus) - window_size // 2 - 1:
            num_after = len(corpus) - i - 1
            num_before = window_size - num_after
        else:
            num_before = window_size // 2
            num_after = window_size // 2

        before = list(range(max(0, i - num_before), i))
        after = list(range(i + 1, min(len(corpus), i + 1 + num_after)))
        context_indices = before + after
        pseudo_doc = [corpus[j] for j in context_indices]

        if not pseudo_doc:
            chunk_result.append((i, []))
            continue

        combined_text = " ".join(pseudo_doc)
        keywords = kw_extractor.extract_keywords(combined_text)
        sorted_keywords = sorted(keywords, key=lambda x: x[1])
        salient_words = [kw for kw, _ in sorted_keywords[:top_k]]
        chunk_result.append((i, salient_words))

    print(f"[Worker {worker_id}] Finished processing {len(chunk_indices)} lines.")
    return chunk_result


def split_indices_evenly(total, num_chunks):
    chunk_size = (total + num_chunks - 1) // num_chunks
    return [list(range(i * chunk_size, min((i + 1) * chunk_size, total))) for i in range(num_chunks)]


def extract_salient_parallel_chunked(corpus, window_size=4, top_k=5, num_workers=4):
    index_chunks = split_indices_evenly(len(corpus), num_workers)
    args = [(chunk, corpus, window_size, top_k, wid) for wid, chunk in enumerate(index_chunks)]

    with mp.Pool(num_workers) as pool:
        results = pool.starmap(extract_yake_chunk, args)

    # Flatten and reorder results
    flattened = [item for chunk in results for item in chunk]
    sorted_results = [salient for _, salient in sorted(flattened, key=lambda x: x[0])]
    return sorted_results


def write_salient_prefixed_raw_file(raw_lines, output_file, salient_word_lists):
    with open(output_file, "w") as f:
        for line, salient_words in zip(raw_lines, salient_word_lists):
            prefixed_line = ' '.join(salient_words + ['__SEP__'] + line.split())
            f.write(prefixed_line + '\n')
    print(f"[INFO] Wrote output to {output_file}")


# Run
if __name__ == "__main__":
    print("[INFO] Loading input...")
    raw_lines = open("en-zh.en-filtered-wsd-processed.en", "r").read().splitlines()
    print(f"[INFO] Loaded {len(raw_lines)} lines.")

    salient_contexts = extract_salient_parallel_chunked(
        raw_lines,
        window_size=4,
        top_k=4,
        num_workers=mp.cpu_count()
    )

    write_salient_prefixed_raw_file(raw_lines, "en-zh.en-filtered-wsd-processed-salient.en", salient_contexts)


[INFO] Loading input...
[INFO] Loaded 224743 lines.
[Worker 4] Finished processing 14047 lines.
[Worker 8] Finished processing 14047 lines.
[Worker 5] Finished processing 14047 lines.
[Worker 6] Finished processing 14047 lines.
[Worker 2] Finished processing 14047 lines.
[Worker 9] Finished processing 14047 lines.
[Worker 7] Finished processing 14047 lines.
[Worker 15] Finished processing 14038 lines.
[Worker 11] Finished processing 14047 lines.
[Worker 1] Finished processing 14047 lines.
[Worker 3] Finished processing 14047 lines.
[Worker 0] Finished processing 14047 lines.
[Worker 14] Finished processing 14047 lines.
[Worker 12] Finished processing 14047 lines.
[Worker 13] Finished processing 14047 lines.
[Worker 10] Finished processing 14047 lines.
[INFO] Wrote output to en-zh.en-filtered-wsd-processed-salient.en


In [31]:
# train a sentencepiece model for subwording
!python MT-Preparation/subwording/1-train_unigram.py ./en-zh.en-filtered-wsd-processed-salient.en ./en-zh.zh-filtered-wsd.zh

sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=./en-zh.en-filtered-wsd-processed-salient.en --model_prefix=source --vocab_size=10000 --hard_vocab_limit=false --split_digits=true --user_defined_symbols=__SEP__,#
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: ./en-zh.en-filtered-wsd-processed-salient.en
  input_format: 
  model_prefix: source
  model_type: UNIGRAM
  vocab_size: 10000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 1
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: __SEP__
  user_defined_symbols: #
  required_chars: 
  byte_fallback: 0
  

In [32]:
# subword the dataset
import sentencepiece as spm

def process_line(line, sp_processor):
    # Separate text and labels
    parts = line.split()
    text_parts = []
    label_parts = []
    for part in parts:
        if "#" in part:  # Assumes labels are marked with a hash (e.g., "word#label")
            label_parts.append(part)
        else:
            text_parts.append(part)

    # Tokenize the text only
    tokenized_text = sp_processor.encode_as_pieces(' '.join(text_parts))
    tokenized_text = " ".join(tokenized_text)

    # Combine tokenized text and labels
    return tokenized_text + " " + " ".join(label_parts)

def subword_tokenize_file(input_path, output_path, sp_model_path):
    sp = spm.SentencePieceProcessor()
    sp.load(sp_model_path)

    with open(input_path, 'r', encoding='utf-8') as infile, \
         open(output_path, 'w+', encoding='utf-8') as outfile:
        for line in infile:
            line = line.strip()
            processed_line = process_line(line, sp)
            outfile.write(processed_line + "\n")

    print("Done subwording the file! Output:", output_path)

# Example usage
source_model_path = 'source.model'
target_model_path = 'target.model'
source_raw_path = './en-zh.en-filtered-wsd-processed-salient.en'
target_raw_path = './en-zh.zh-filtered-wsd.zh'
source_subworded_path = source_raw_path + ".subword"
target_subworded_path = target_raw_path + ".subword"

subword_tokenize_file(source_raw_path, source_subworded_path, source_model_path)
subword_tokenize_file(target_raw_path, target_subworded_path, target_model_path)


Done subwording the file! Output: ./en-zh.en-filtered-wsd-processed-salient.en.subword
Done subwording the file! Output: ./en-zh.zh-filtered-wsd.zh.subword


In [33]:
# first 3 lines before subwording
!head -n 3 ./en-zh.en-filtered-wsd-processed-salient.en && echo "-----" && head -n 3 ./en-zh.zh-filtered-wsd.zh



Chris great honor extremely __SEP__ en
blown comments night conference __SEP__ Thank you so#1 much#1, Chris . And it's truly#1 a#1 great#1 honor#1 to have#1 the opportunity to come#1 to this stage#1 twice#1; I#1'm extremely#1 grateful#1.
Chris great honor extremely __SEP__ I#1 have#1 been#1 blown#1 away#1 by#1 this conference#1, and I want#1 to thank all#1 of you for the many nice#1 comments#1 about#1 what I had#1 to say#1 the other#1 night#1.
-----
zh
非常谢谢，克里斯。的确非常荣幸 能有第二次站在这个台上的机会，我真是非常感激。
这个会议真是让我感到惊叹不已，我还要谢谢你们留下的 关于我上次演讲的精彩评论


In [35]:
# first 3 lines after subwording
!head -n 3 ./en-zh.en-filtered-wsd-processed-salient.en.subword && echo "---" && head -n 3 ./en-zh.zh-filtered-wsd.zh.subword

▁Chris ▁great ▁honor ▁extremely ▁ __SEP__ ▁ en 
▁ blown ▁comment s ▁night ▁conference ▁ __SEP__ ▁Thank ▁you ▁Chris ▁ . ▁And ▁it ' s ▁to ▁the ▁opportunity ▁to ▁to ▁this so#1 much#1, truly#1 a#1 great#1 honor#1 have#1 come#1 stage#1 twice#1; I#1'm extremely#1 grateful#1.
▁Chris ▁great ▁honor ▁extremely ▁ __SEP__ ▁this ▁and ▁I ▁to ▁thank ▁of ▁you ▁for ▁the ▁many ▁what ▁I ▁to ▁the I#1 have#1 been#1 blown#1 away#1 by#1 conference#1, want#1 all#1 nice#1 comments#1 about#1 had#1 say#1 other#1 night#1.
---
▁ z h 
▁非常 谢谢 , 克里斯 。 的确 非常 荣幸 ▁能 有 第二次 站在 这个 台上 的机会 , 我 真是 非常 感激 。 
▁这个 会议 真是 让我 感到 惊 叹 不 已 , 我 还要 谢谢你们 留下 的 ▁关于 我 上 次 演讲 的 精彩 评论 


In [36]:
# split the dataset into training set, development set, and test set
# Development and test sets should be between 100 and 500 segments (here we chose 200)
!python3 MT-Preparation/train_dev_split/train_dev_test_split.py 2000 2000 ./en-zh.en-filtered-wsd-processed-salient.en.subword ./en-zh.zh-filtered-wsd.zh.subword

Dataframe shape: (224743, 2)
--- Empty Cells Deleted --> Rows: 224743
--- Wrote Files
Done!
Output files
./en-zh.en-filtered-wsd-processed-salient.en.subword.train
./en-zh.zh-filtered-wsd.zh.subword.train
./en-zh.en-filtered-wsd-processed-salient.en.subword.dev
./en-zh.zh-filtered-wsd.zh.subword.dev
./en-zh.en-filtered-wsd-processed-salient.en.subword.test
./en-zh.zh-filtered-wsd.zh.subword.test


In [37]:
!wc -l ./*.subword.*

     2000 ./en-zh.en-filtered-wsd-processed-salient.en.subword.dev
     2000 ./en-zh.en-filtered-wsd-processed-salient.en.subword.test
   220743 ./en-zh.en-filtered-wsd-processed-salient.en.subword.train
     2000 ./en-zh.en-filtered-wsd-processed.en.subword.dev
     2000 ./en-zh.en-filtered-wsd-processed.en.subword.test
   220743 ./en-zh.en-filtered-wsd-processed.en.subword.train
     2000 ./en-zh.en-filtered-wsd.en.subword.dev
     2000 ./en-zh.en-filtered-wsd.en.subword.test
     2000 ./en-zh.en-filtered-wsd.en.subword.test.desubword
   220743 ./en-zh.en-filtered-wsd.en.subword.train
     2000 ./en-zh.zh-filtered-wsd.zh.subword.dev
     2000 ./en-zh.zh-filtered-wsd.zh.subword.test
     2000 ./en-zh.zh-filtered-wsd.zh.subword.test.desubword
   220743 ./en-zh.zh-filtered-wsd.zh.subword.train
   902972 total


In [38]:
# check the first and last line from each dataset
!echo "---First line---"
!head -n 1 ./*.{train,dev,test}

!echo -e "\n---Last line---"
!tail -n 1 ./*.{train,dev,test}

---First line---
==> ./en-zh.en-filtered-wsd-processed-salient.en.subword.train <==
▁Chris ▁great ▁honor ▁extremely ▁ __SEP__ ▁ en 

==> ./en-zh.en-filtered-wsd-processed.en.subword.train <==
▁ en 

==> ./en-zh.en-filtered-wsd.en.subword.train <==
▁en

==> ./en-zh.zh-filtered-wsd.zh.subword.train <==
▁ z h 

==> ./en-zh.en-filtered-wsd-processed-salient.en.subword.dev <==
▁States ▁rural ▁variable ▁important ▁ __SEP__ ▁it ' s ▁the ▁of ▁these ▁it ' s ▁and ▁the ▁of ▁that ▁you ▁which ▁we ' ll So#1 combination#1 two#1 things#1: education#1 type#1 neighbors#1 have#1, talk#1 about#1 more#1 in#1 a#1 moment#1.

==> ./en-zh.en-filtered-wsd-processed.en.subword.dev <==
▁it ' s ▁the ▁of ▁these ▁it ' s ▁and ▁the ▁of ▁that ▁you ▁which ▁we ' ll So#1 combination#1 two#1 things#1: education#1 type#1 neighbors#1 have#1, talk#1 about#1 more#1 in#1 a#1 moment#1.

==> ./en-zh.en-filtered-wsd.en.subword.dev <==
▁So ▁it ' s ▁the ▁combination ▁of ▁these ▁two ▁things : ▁it ' s ▁education ▁and ▁the ▁type ▁of ▁n